## About this notebook:

This notebook uses saved transformations to align all channels in a list of wells.

Input:
- a set of transforms (one transform/round)
- data frames with channel data for each channel
- nd2 images

Output:
- aligned tiffs 
- updated data frames

## Choose the experiment

Set `CONFIG` to the experiment's `experiment.json`. All folders and the well list come from it; create or edit it with `setup_experiment.ipynb`.

In [ ]:
from fouri import load_config

# This experiment's experiment.json (create or edit it with setup_experiment.ipynb)
CONFIG = r'Z:\PATH\TO\EXPERIMENT\experiment.json'

exp = load_config(CONFIG)

path_df   = exp.output_df      # data frames (output of 00)
path_tmat = path_df            # transformations (output of 02/03)
path_save = exp.aligned_tiffs  # where to save aligned tiffs
well_list = exp.well_list

## Prepare for processing

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np

from tifffile import imwrite
from skimage import transform

from fouri import make_dir, read_nd2_channel

In [ ]:
# create directory for saving aligned tiffs if needed
make_dir(path_save, 'aligned tiffs')

# Align and save images

In [ ]:
for selected_well in well_list:

    print(f'Processing well {selected_well}.')

    # create output directory
    make_dir(os.path.join(path_save,selected_well), f'{selected_well} aligned tiffs')

    # open data frame
    df_name = f'df_{selected_well}.pkl'

    #myData = pd.read_pickle(os.path.join(path_df,df_name))
    with open(os.path.join(path_df,df_name), "rb") as fh: # in case of protocol 5 issues
        myData = pickle.load(fh)

    # open transformations
    tmat_name = f'tmat_{selected_well}.pkl'
    tmat_list = pickle.load(open(os.path.join(path_tmat,tmat_name), "rb"))

    for ind,my_signal in myData.iterrows():

        # get a path to the image file
        file_path = os.path.join(my_signal.dir,my_signal.sub_dir,my_signal.file)

        # read the right channel (the nd2 file is closed afterwards)
        im = read_nd2_channel(file_path, int(my_signal.channel_in_file))

        # trim if needed
        if ((my_signal.width_min < my_signal.width) | (my_signal.height_min < my_signal.height)):

            im = im[:int(my_signal.height_min),:int(my_signal.width_min)]

        # get the transform
        tf = tmat_list[int(my_signal.alignRound)]

        # apply transform
        im_alig = transform.warp(im,tf,output_shape=im.shape)
        im_alig[im_alig<0] = 0

        # get the range correct
        im_alig = im_alig*2**16

        # save image
        #alig_name = f'Round_{str(int(my_signal.nameRound)).zfill(2)}_well{my_signal.well}_{my_signal.signal}.tif'
        #Removed the int tag from my_signal.nameRound so that the new naming scheme could be parsed. Still avoiding leading zeros
        alig_name = f'Round_{str(my_signal.nameRound).zfill(2)}_well{my_signal.well}_{my_signal.signal}.tif'
        imwrite(os.path.join(path_save,selected_well,alig_name),im_alig.astype('uint16'))

        # remember this in the df
        myData.loc[ind,'aligned_file_name'] = alig_name

    # save df
    myData.to_pickle(os.path.join(path_df,df_name))